# 02 · Preprocesamiento (resultados de la etapa 4)

Muestra qué hizo `python -m motor --etapas 4_preprocesamiento` con duplicados, valores imposibles, faltantes, outliers y balance. Las reglas están en `motor/e4_preprocesamiento.py` y en el LEEME.

In [1]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

# Raíz del paquete: la carpeta que contiene config/motor.json
RAIZ = Path.cwd()
while not (RAIZ / "config" / "motor.json").is_file() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SAL = Path(os.environ.get("CLASIFICADOR_SALIDAS", RAIZ / "salidas"))
ENT = RAIZ / "datos_entrada" / "processed"
TY = RAIZ / "resultados_TY" / "run_004"
if not (TY / "02_particion" / "particion.parquet").is_file():
    TY = RAIZ / "resultados_TY" / "run_003"   # respaldo: corrida anterior incluida en el paquete
print("raíz:", RAIZ, "| salidas:", SAL, "| TY:", TY.name)

raíz: /home/claude/build/Modelo | salidas: /home/claude/build/Modelo/salidas | TY: run_004


In [2]:
P = SAL / "4_preprocesamiento"
pre = json.loads((P / "preprocesamiento.json").read_text(encoding="utf-8"))
print(json.dumps(pre, indent=1, ensure_ascii=False))

{
 "filas_inicio": 11850,
 "filas_final": 11815,
 "quitadas_duplicado": 35,
 "valores_imposibles_a_faltante": {},
 "variables_quitadas_por_faltantes": [],
 "variables_por_conjunto": {
  "catalogo": 21,
  "catalogo_mas_spherex": 28,
  "servible": 10
 },
 "outliers_marcados_total": 1107,
 "con_spherex": true,
 "ty_run": "/workspace/CLASIFICADOR_VAST/resultados_TY/run_004"
}


## Duplicados

`posicion`: mismo objeto físico con dos `oid` (a menos de 2″ tras propagar a 2016.0). `vector_identico`: mismas variables físicas. Si las etiquetas difieren, se quitan todos.

In [3]:
d = pd.read_csv(P / "duplicados.csv")
print(d["accion"].value_counts()); d.head(20)

accion
conservar                       54
quitar_duplicado                54
quitar_etiqueta_en_conflicto     4
Name: count, dtype: int64


,tipo,grupo,oid,label,accion
0,posicion,2977,2977,M,conservar
1,posicion,2977,2978,M,quitar_duplicado
2,posicion,3542,3542,M,quitar_etiqueta_en_conflicto
3,posicion,3542,3543,L,quitar_etiqueta_en_conflicto
4,posicion,3576,3576,M,conservar
5,posicion,3576,3577,M,quitar_duplicado
6,posicion,4032,4032,star,conservar
7,posicion,4032,4222,star,quitar_duplicado
8,posicion,4301,4301,star,conservar
9,posicion,4301,4331,star,quitar_duplicado


## Faltantes por clase (desarrollo)

No se imputan aquí: se imputan dentro de cada pliegue con la mediana del entrenamiento.

In [4]:
pd.read_csv(P / "faltantes_por_clase.csv", index_col=0)

,L,M,T,Y,star,total_dev
variable,,,,,,
cw_W1_W2,0.0928,0.1861,0.1646,0.225,0.1997,0.1783
aw_W1_W2,0.1139,0.1889,0.3293,0.775,0.3723,0.3136
uw_W1_W2,0.0813,0.2038,0.2816,0.950,0.2022,0.1916
J_H,0.1862,0.1141,0.7782,1.000,0.3240,0.3230
H_Ks,0.1811,0.1128,0.8463,1.000,0.3869,0.3698
J_Ks,0.1951,0.1236,0.8476,1.000,0.3868,0.3730
J_W1,0.2348,0.2160,0.7578,1.000,0.3764,0.3728
J_W2,0.2354,0.2201,0.7605,1.000,0.3883,0.3815
H_W2,0.2214,0.2188,0.7959,1.000,0.3995,0.3893


## Outliers por clase (se informan, no se borran)

In [5]:
o = pd.read_csv(P / "outliers_por_clase.csv")
o.sort_values("frac_outlier", ascending=False).head(20)

,variable,clase,n,mediana,mad,n_outlier,frac_outlier
86,cw_mu_snr,Y,32,15.528683,10.917891,2,0.062500
10,uw_W1_W2,M,586,0.224779,0.059636,29,0.049488
85,cw_mu_snr,T,655,11.828822,13.296196,32,0.048855
70,W1_W3,M,514,0.527500,0.203116,22,0.042802
83,cw_mu_snr,L,1522,16.324561,17.248645,63,0.041393
68,W2_W3,star,832,0.306000,0.510014,33,0.039663
66,W2_W3,M,514,0.289500,0.173464,20,0.038911
72,W1_W3,star,832,0.273000,0.524099,28,0.033654
84,cw_mu_snr,M,694,38.892197,42.716979,23,0.033141
1,cw_W1_W2,M,599,0.240000,0.044478,19,0.031720


## Balance de clases

In [6]:
print(json.dumps(json.loads((P / "balance.json").read_text(encoding="utf-8")), indent=1, ensure_ascii=False))

{
 "conteos": {
  "5clases": {
   "dev": {
    "star": 6371,
    "L": 1563,
    "M": 736,
    "T": 735,
    "Y": 40
   },
   "test": {
    "star": 1599,
    "L": 394,
    "M": 184,
    "T": 183,
    "Y": 10
   },
   "dev_subconjunto_spherex": {
    "T": 251,
    "L": 167,
    "star": 160,
    "M": 79,
    "Y": 40
   }
  },
  "servible": {
   "dev": {
    "star": 8670,
    "cold_brown_dwarf": 775
   },
   "test": {
    "star": 2177,
    "cold_brown_dwarf": 193
   },
   "dev_subconjunto_spherex": {
    "star": 406,
    "cold_brown_dwarf": 291
   }
  }
 },
 "estrategia": "pesos por clase (N/(K·n_c))^p, p elegido por CV entre [0.25]; sin remuestreo ni SMOTE"
}


## Tabla final para modelar

In [7]:
tab = pd.read_parquet(P / "tabla_modelo.parquet")
var = json.loads((P / "variables.json").read_text(encoding="utf-8"))
print(tab.shape); print({k: len(v) for k, v in var["conjuntos"].items()})
print("auditoría (no predictores):", var["auditoria_no_predictor"])
tab.head()

(11815, 38)
{'catalogo': 21, 'catalogo_mas_spherex': 28, 'servible': 10}
auditoría (no predictores): ['spx_snr_max', 'spx_n_det', 'spx_n_ok', 'spx_ch4_limite']


,oid,label,role,cv_fold,sky_cell,in_spherex,cw_W1_W2,aw_W1_W2,uw_W1_W2,J_H,H_Ks,J_Ks,J_W1,J_W2,H_W2,Ks_W2,cw_mu,cw_log_mu,Hrpm_W2,Hrpm_J,aw_var_w2,W2_W3,W1_W3,W3_detectada,cw_W1_W2_pm,cw_mu_snr,Hrpm_W2_pm,spx_c21,spx_c32,spx_c43,spx_c54,spx_c65,spx_c64,spx_ch4,spx_snr_max,spx_n_det,spx_n_ok,spx_ch4_limite
0,0,T,test,-1,168,False,0.008,NaN,0.085621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.033191,-1.478982,14.474092,NaN,<NA>,NaN,NaN,0,-0.107,0.598597,14.577092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,T,dev,0,192,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,M,test,-1,168,False,0.207,0.276,0.213185,0.593,0.561,1.154,1.308,1.515,0.922,0.361,0.135114,-0.869301,14.169497,15.684497,1,1.237,1.513,1,0.218,20.016299,14.150497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,L,test,-1,168,False,0.410,0.426,0.389268,1.017,0.795,1.812,2.555,2.965,1.948,1.153,0.231759,-0.634964,14.382182,17.347182,0,0.867,1.293,1,0.403,45.890593,14.377182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,L,dev,3,120,True,0.287,0.351,0.261938,NaN,0.336,NaN,NaN,NaN,0.967,0.631,0.483909,-0.315236,17.490819,NaN,0,NaN,NaN,0,0.289,50.912411,17.444819,NaN,-0.339648,0.54035,-0.021527,0.761196,0.73967,0.008611,37.408059,4.0,17.0,0.0
